# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "documents/managing_oneself.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(len(docs), "pages loaded")
print(document_text[:500])


13 pages loaded
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
from pydantic import BaseModel, Field
from openai import OpenAI

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="<= 1 paragraph: why this is relevant for an AI professional's professional development")
    Summary: str = Field(description="Concise summary no longer than 1000 tokens")
    Tone: str
    InputTokens: int
    OutputTokens: int

TONE = "Formal Academic Writing"

developer_prompt = """
You are an expert analytical assistant.

Tasks:
1. Identify the author and title.
2. Write a relevance statement (<= 1 paragraph) explaining why this article matters for an AI professional's professional development.
3. Write a concise summary no longer than 1000 tokens.
4. Maintain this tone throughout: {tone}

Return output strictly matching the provided schema.
"""

user_prompt = "Here is the article:\n\n{context}"


In [ ]:
import os

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",  
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)


In [9]:
response = client.responses.parse(
    model="gpt-4o-mini",  
    input=[
        {"role": "developer", "content": developer_prompt.format(tone=TONE)},
        {"role": "user", "content": user_prompt.format(context=document_text)}
    ],
    text_format=ArticleSummary
)

summary_output = response.output_parsed
summary_output.InputTokens = response.usage.input_tokens
summary_output.OutputTokens = response.usage.output_tokens
summary_output.Tone = TONE

summary_output.model_dump()


{'Author': 'Peter F. Drucker',
 'Title': 'Managing Oneself',
 'Relevance': "This article is pivotal for AI professionals as it emphasizes self-awareness in career management. Understanding one's strengths, work styles, and values empowers professionals in a rapidly evolving tech landscape to navigate their careers effectively, thereby enhancing both personal satisfaction and organizational contribution.",
 'Summary': 'In "Managing Oneself," Peter F. Drucker asserts that knowledge workers must assume responsibility for their careers, acting as their own chief executive officers. He outlines critical self-reflective questions to help individuals identify their strengths, preferred work styles, and core values. Drucker emphasizes the necessity of understanding how one learns and interacts with others to maximize contributions in the workplace. Through feedback analysis, individuals can discern their effective performance areas while avoiding unproductive habits. He also discusses the impo

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:

import os
from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_output.Summary
)


summarization_questions = [
    "Does the summary accurately reflect Drucker’s core thesis about managing oneself?",
    "Does the summary correctly represent the role of strengths (e.g., learning how you perform)?",
    "Does the summary capture Drucker’s argument about values and ethical commitments in work choices?",
    "Does the summary include the idea of contribution (what results to aim for) and professional fit?",
    "Does the summary avoid adding claims that are not supported by the article?"
]

summ_metric = SummarizationMetric(
    assessment_questions=summarization_questions,
    include_reason=True,
    model=model
)

summ_metric.measure(test_case)


coherence_questions = [
    "Is the summary logically organized from point to point?",
    "Are transitions between ideas clear and smooth?",
    "Is the summary easy to follow for a reader unfamiliar with the article?",
    "Does the summary avoid redundancy or repetition?",
    "Are key concepts explained clearly without ambiguity?"
]

coherence_metric = GEval(
    name="Coherence",
    criteria="\n".join([f"- {q}" for q in coherence_questions]),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

coherence_metric.measure(test_case)


tonality_questions = [
    "Does the summary maintain a formal academic tone?",
    "Is the tone consistent throughout the summary?",
    "Is diction analytical and professional (not conversational)?",
    "Does the summary avoid slang, casual phrasing, or informality?",
    "Is the tone clearly recognizable as 'Formal Academic Writing'?"
]

tonality_metric = GEval(
    name="Tonality",
    criteria="\n".join([f"- {q}" for q in tonality_questions]),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric.measure(test_case)


safety_questions = [
    "Does the summary avoid hallucinated facts about author/publication/claims?",
    "Does the summary avoid misleading distortions of the article’s argument?",
    "Does the summary avoid harmful, biased, or discriminatory language?",
    "Does the summary avoid unsafe recommendations (medical/legal/financial)?",
    "Is the summary faithful to the source text overall?"
]

safety_metric = GEval(
    name="Safety",
    criteria="\n".join([f"- {q}" for q in safety_questions]),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric.measure(test_case)


evaluation_results = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

evaluation_results


Output()

Output()

Output()

Output()

{'SummarizationScore': 0.75,
 'SummarizationReason': 'The score is 0.75 because the summary introduces extra information that is not present in the original text, such as specific self-reflective questions and the need for continual adaptation in a changing job market. However, it does not contradict any information from the original text, which maintains a level of accuracy.',
 'CoherenceScore': 0.8562176500885798,
 'CoherenceReason': "The summary effectively captures the main points of Drucker's article, presenting a logical structure that aligns with the key themes of self-management, self-reflection, and the importance of aligning personal and organizational values. Transitions between ideas are smooth, enhancing coherence. The response is clear and comprehensible for readers unfamiliar with the original article, explaining complex concepts in a straightforward manner. However, it could benefit from slightly more detail on specific methods, such as feedback analysis, to further cla

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [14]:
enhancement_prompt = """
You previously generated a summary of an article.

Original context:
{context}

Previous summary:
{summary}

Evaluation feedback:
{evaluation}

Task:
Produce an improved version of the summary that:
- Fixes weaknesses highlighted in the evaluation
- Preserves factual accuracy
- Improves coherence and clarity
- Strictly maintains this tone: {tone}

Return only the revised summary.
"""


In [15]:
improved_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "developer",
            "content": enhancement_prompt.format(
                context=document_text,
                summary=summary_output.Summary,
                evaluation=evaluation_results,
                tone=TONE
            )
        }
    ]
)

improved_summary = improved_response.output_text
print(improved_summary[:500])


In "Managing Oneself," Peter F. Drucker emphasizes that contemporary knowledge workers must take personal responsibility for their careers, positioning themselves as their own chief executive officers. He introduces a series of self-reflective questions essential for identifying individual strengths, preferred work styles, core values, and optimal work environments. Drucker highlights the significance of understanding one’s learning preferences and interpersonal dynamics in order to enhance work


In [16]:
from deepeval.test_case import LLMTestCase

improved_test_case = LLMTestCase(
    input=document_text,
    actual_output=improved_summary
)


In [17]:
summ_metric.measure(improved_test_case)
coherence_metric.measure(improved_test_case)
tonality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)


Output()

Output()

Output()

Output()

0.8798186770538576

In [18]:
improved_evaluation_results = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

improved_evaluation_results


{'SummarizationScore': 0.7272727272727273,
 'SummarizationReason': 'The score is 0.73 because the summary includes extra information that was not present in the original text, which may lead to misinterpretation of the original content. However, there are no contradictions, and the summary captures the essence of the original text.',
 'CoherenceScore': 0.8731058584489497,
 'CoherenceReason': "The summary effectively captures the logical structure of Drucker's main points, including the importance of self-management and feedback analysis. Transitions between ideas are smooth, enhancing coherence. The response is clear and comprehensible for readers unfamiliar with the original article, explaining key concepts without ambiguity. However, it could benefit from slightly more detail on specific examples or implications of Drucker's ideas to achieve a perfect score.",
 'TonalityScore': 0.8268459085862456,
 'TonalityReason': "The response maintains a formal tone consistent with academic writi

In [19]:
print("Original Scores:")
print(evaluation_results)

print("\nImproved Scores:")
print(improved_evaluation_results)


Original Scores:
{'SummarizationScore': 0.75, 'SummarizationReason': 'The score is 0.75 because the summary introduces extra information that is not present in the original text, such as specific self-reflective questions and the need for continual adaptation in a changing job market. However, it does not contradict any information from the original text, which maintains a level of accuracy.', 'CoherenceScore': 0.8562176500885798, 'CoherenceReason': "The summary effectively captures the main points of Drucker's article, presenting a logical structure that aligns with the key themes of self-management, self-reflection, and the importance of aligning personal and organizational values. Transitions between ideas are smooth, enhancing coherence. The response is clear and comprehensible for readers unfamiliar with the original article, explaining complex concepts in a straightforward manner. However, it could benefit from slightly more detail on specific methods, such as feedback analysis, 

Please, do not forget to add your comments.

# The enhancement prompt aimed to address the evaluation-identified weaknesses by improving clarity, coherence, and tone while staying loyal to the source text. The revised summary shows mixed results where coherence and tonality scores increased slightly while summarization scored decreased. These results even though slightly improved shows the interpretive tendency of these models. Overall the enhancement improved the quality slightly but we cannot say it improved the content accuracy across the board. 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
